In [40]:
# import library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

### Load Data

In [41]:
# load data
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

In [42]:
train.shape

(1460, 81)

### Data Preprocessing

In [43]:
train.isna().sum()

Id                  0
MSSubClass          0
MSZoning            0
LotFrontage       259
LotArea             0
Street              0
Alley            1369
LotShape            0
LandContour         0
Utilities           0
LotConfig           0
LandSlope           0
Neighborhood        0
Condition1          0
Condition2          0
BldgType            0
HouseStyle          0
OverallQual         0
OverallCond         0
YearBuilt           0
YearRemodAdd        0
RoofStyle           0
RoofMatl            0
Exterior1st         0
Exterior2nd         0
MasVnrType        872
MasVnrArea          8
ExterQual           0
ExterCond           0
Foundation          0
BsmtQual           37
BsmtCond           37
BsmtExposure       38
BsmtFinType1       37
BsmtFinSF1          0
BsmtFinType2       38
BsmtFinSF2          0
BsmtUnfSF           0
TotalBsmtSF         0
Heating             0
HeatingQC           0
CentralAir          0
Electrical          1
1stFlrSF            0
2ndFlrSF            0
LowQualFin

In [44]:
# cat column
cat_col = train.select_dtypes(include=['object']).columns

# NA column
na_col = train.columns[train.isna().sum() > 0]

# NA column with meaning (from the description)
na_col_meaning = ["Alley", "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2", "FireplaceQu", "GarageType", "GarageFinish", "GarageQual", "GarageCond", "PoolQC", "Fence", "MiscFeature"]

# NA column meaning - all NA col
na_col_no_meaning = [col for col in na_col if col not in na_col_meaning]

In [45]:
na_col_no_meaning

['LotFrontage', 'MasVnrType', 'MasVnrArea', 'Electrical', 'GarageYrBlt']

In [46]:
# drop Utilities column (only 1 value)
train.drop(columns=["Utilities"], inplace=True)
test.drop(columns=["Utilities"], inplace=True)

In [47]:
# by hand split categorical columns into 3 types
ordinal_cols = ["LotShape","LandContour","LandSlope","HouseStyle","ExterQual","ExterCond","BsmtQual","BsmtCond","BsmtExposure","BsmtFinType1","BsmtFinType2","HeatingQC","KitchenQual","Functional","FireplaceQu","GarageFinish","GarageQual","GarageCond","PavedDrive","PoolQC","Fence"]
cat_cols = ["Street","Alley","CentralAir","BldgType","SaleType","SaleCondition","MSSubClass","MSZoning","Neighborhood","Condition1","Condition2","LotConfig","RoofStyle","RoofMatl","Exterior1st","Exterior2nd","MasVnrType","Foundation","Heating","Electrical","GarageType","MiscFeature"]

threshold = 4 
one_hot_cols = []
freq_cols = []
for col in cat_cols:
    if col in train.columns and train[col].nunique() <= threshold:
        one_hot_cols.append(col)
    else:
        freq_cols.append(col)

In [48]:
# ! this is removed because it changed to use the KNN at the end
# from sklearn.impute import SimpleImputer
# categorical_na = ['MasVnrType', 'Electrical']
# if categorical_na:
#     cat_imputer = SimpleImputer(strategy='most_frequent')
#     train[categorical_na] = cat_imputer.fit_transform(train[categorical_na])
#     test[categorical_na] = cat_imputer.transform(test[categorical_na])

In [49]:
from sklearn.preprocessing import OrdinalEncoder

# define categories in order (this is hard code, using the data_description.tx)
ordinal_categories = [
    ['Reg', 'IR1', 'IR2', 'IR3'],  # LotShape
    ['Lvl', 'Bnk', 'HLS', 'Low'],  # LandContour
    # ['AllPub', 'NoSewr', 'NoSeWa', 'ELO'],  # Utilities
    ['Gtl', 'Mod', 'Sev'],  # LandSlope
    ['1Story', '1.5Fin', '1.5Unf', '2Story', '2.5Fin', '2.5Unf', 'SFoyer', 'SLvl'],  # HouseStyle
    ['Ex', 'Gd', 'TA', 'Fa', 'Po'],  # ExterQual
    ['Ex', 'Gd', 'TA', 'Fa', 'Po'],  # ExterCond
    ['Ex', 'Gd', 'TA', 'Fa', 'Po', 'NA'],  # BsmtQual
    ['Ex', 'Gd', 'TA', 'Fa', 'Po', 'NA'],  # BsmtCond
    ['Gd', 'Av', 'Mn', 'No', 'NA'],  # BsmtExposure
    ['GLQ', 'ALQ', 'BLQ', 'Rec', 'LwQ', 'Unf', 'NA'],  # BsmtFinType1
    ['GLQ', 'ALQ', 'BLQ', 'Rec', 'LwQ', 'Unf', 'NA'],  # BsmtFinType2
    ['Ex', 'Gd', 'TA', 'Fa', 'Po'],  # HeatingQC
    ['Ex', 'Gd', 'TA', 'Fa', 'Po'],  # KitchenQual
    ['Typ', 'Min1', 'Min2', 'Mod', 'Maj1', 'Maj2', 'Sev', 'Sal'],  # Functional
    ['Ex', 'Gd', 'TA', 'Fa', 'Po', 'NA'],  # FireplaceQu
    ['Fin', 'RFn', 'Unf', 'NA'],  # GarageFinish
    ['Ex', 'Gd', 'TA', 'Fa', 'Po', 'NA'],  # GarageQual
    ['Ex', 'Gd', 'TA', 'Fa', 'Po', 'NA'],  # GarageCond
    ['Y', 'P', 'N'],  # PavedDrive
    ['Ex', 'Gd', 'TA', 'Fa', 'NA'],  # PoolQC
    ['GdPrv', 'MnPrv', 'GdWo', 'MnWw', 'NA']  # Fence
]

# fit encoder on train (handle NA by filling with 'NA' first)
for col in ordinal_cols:
    if col in train.columns:
        cat_list = ordinal_categories[ordinal_cols.index(col)]
        if 'NA' in cat_list:
            train[col] = train[col].fillna('NA')
            test[col] = test[col].fillna('NA')
        else:
            mode = train[col].mode()[0]
            train[col] = train[col].fillna(mode)
            test[col] = test[col].fillna(mode)

encoder = OrdinalEncoder(categories=ordinal_categories)
train[ordinal_cols] = encoder.fit_transform(train[ordinal_cols])
test[ordinal_cols] = encoder.transform(test[ordinal_cols])

In [50]:
# one hot encoding for column with few category
train = pd.get_dummies(train, columns=one_hot_cols, drop_first=True)
test = pd.get_dummies(test, columns=one_hot_cols, drop_first=True)

missing_cols = set(train.columns) - set(test.columns)
for col in missing_cols:
    test[col] = 0
extra_cols = set(test.columns) - set(train.columns)
test = test.drop(columns=extra_cols)
test = test[train.columns]

missing_cols = set(train.columns) - set(test.columns)
for col in missing_cols:
    test[col] = 0
extra_cols = set(test.columns) - set(train.columns)
test = test.drop(columns=extra_cols)

In [51]:
# frequency encoding for column with many category
for col in freq_cols:
    if col in train.columns:
        freq_map = train[col].value_counts(normalize=True)
        train[col + '_freq'] = train[col].map(freq_map)
        test[col + '_freq'] = test[col].map(freq_map).fillna(0)
        train.drop(columns=[col], inplace=True)
        test.drop(columns=[col], inplace=True)

In [52]:
print("train shape after encoding:", train.shape)
print("test shape after encoding:", test.shape)

train shape after encoding: (1460, 83)
test shape after encoding: (1459, 83)


In [53]:
# combine bathroom columns (reduce dimensionality)
train['No. of bathrooms']=train['BsmtFullBath']+0.5*train['BsmtHalfBath']+train['FullBath']+0.5*train['HalfBath']
test['No. of bathrooms']=test['BsmtFullBath']+0.5*test['BsmtHalfBath']+test['FullBath']+0.5*test['HalfBath']
train.drop(columns=['BsmtFullBath','BsmtHalfBath','FullBath','HalfBath'],inplace=True)
test.drop(columns=['BsmtFullBath','BsmtHalfBath','FullBath','HalfBath'],inplace=True)

In [54]:
# filling missing value using KNN method
from sklearn.impute import KNNImputer
knn_imputer = KNNImputer(n_neighbors=5)  # adjust k as needed
train = pd.DataFrame(knn_imputer.fit_transform(train), columns=train.columns)
test = pd.DataFrame(knn_imputer.transform(test), columns=test.columns)

In [55]:
# reorder
order = ['Id', 'LotFrontage', 'LotArea', 'LotShape', 'LandContour', 'LandSlope', 'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'HeatingQC', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual', 'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'No. of bathrooms', 'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual', 'GarageCond', 'PavedDrive', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'PoolQC', 'Fence', 'MiscVal', 'MoSold', 'YrSold', 'SalePrice', 'Street_Pave', 'Alley_Pave', 'CentralAir_Y', 'MasVnrType_BrkFace', 'MasVnrType_Stone', 'MiscFeature_Othr', 'MiscFeature_Shed', 'MiscFeature_TenC', 'BldgType_freq', 'SaleType_freq', 'SaleCondition_freq', 'MSSubClass_freq', 'MSZoning_freq', 'Neighborhood_freq', 'Condition1_freq', 'Condition2_freq', 'LotConfig_freq', 'RoofStyle_freq', 'RoofMatl_freq', 'Exterior1st_freq', 'Exterior2nd_freq', 'Foundation_freq', 'Heating_freq', 'Electrical_freq', 'GarageType_freq']
train = train.loc[:,order]
test = test.loc[:,order]
train['Id'] = train['Id'].round().astype(int)
test['Id'] = test['Id'].round().astype(int)

In [56]:
train.to_csv('data/train_cleaned.csv', index=False)
test.to_csv('data/test_cleaned.csv', index=False)